# Option 2 — Last Granularity Attempt (before the router pivot)

**The question:** does task specialization exist at a *finer* scale than we
tested, or in *contribution space* rather than *activation space*?

**Context:** M1 proved strong per-token sparsity. M2 (contiguous blocks) and
M3 (co-activation clustering, k=38..304, both metrics) found **zero**
task-specialization. This notebook is the FINAL granularity test — if it's
null too, we pivot to token-level routing with three rigorous nulls as
evidence (§55 discipline: no fourth attempt).

**Two arms:**

| arm | what it tests | cost |
|---|---|---|
| **2a — finer k** | k ∈ {608, 1216, 2432} (16/8/4 neurons per expert) + a token-budget sanity check | free (re-clusters cached activations) |
| **2b — token-space clustering** | cluster TOKENS by their 76-dim contribution vector; do token clusters align with task categories? (pure mutual-information test, no neuron partition at all) | free (uses cached z + weights) |

Arm 2b is the important one: it bypasses the neuron-partition question entirely
and asks directly "does the *routing pattern* of a token identify its task?"

---

## The process flow of THIS experiment

```text
 Step 1  environment + archive folder                       ~1 s
 Step 2  knobs                                              ~1 s
 Step 3  load cached captures from M3 (no model!)           ~5 s
 Step 4  arm 2a: finer-k clustering + entropy curve         ~2 min
         + token-budget check (clusters with <min tokens are unreliable)
 Step 5  arm 2b: contribution vectors per token             ~2 min
         (needs down_proj weights -> brief model touch)
 Step 6  arm 2b: cluster tokens, task-vs-cluster MI         ~1 min
 Step 7  verdict + save EVERYTHING to results archive       ~5 s
```

All outputs land in `results/m3b_granularity_option2/` (append-only archive).

## Step 1 — Environment + archive folder

All outputs from this run go to `results/m3b_granularity_option2/` — the
append-only paper archive. Nothing gets deleted, ever (see
`results/ARCHIVE_POLICY.md`).

In [ ]:
import os, sys, json, csv
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path(r"E:\Project\SliceWeave")
sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", r"E:\hf_cache")

M3B = ROOT / "results" / "m3b_granularity_option2"
for sub in ("csv", "plots", "json"):
    (M3B / sub).mkdir(parents=True, exist_ok=True)
CACHE_DIR = ROOT / "results" / "traces" / "m3_capture"   # M3's cached captures

RUN_STAMP = datetime.now().strftime("%Y-%m-%d %H:%M")
print("archive folder:", M3B)
print("run stamp     :", RUN_STAMP)

**What just happened:** archive folder created; run stamp recorded for the manifest. Next: knobs.

## Step 2 — Knobs

Fine k values go down to 4 neurons/expert. The token-budget check in Step 4
will flag any k where clusters are too small for reliable entropy estimates.

In [ ]:
LAYERS        = ["model.layers.7", "model.layers.18", "model.layers.29"]
FINE_KS       = [608, 1216, 2432]     # 16 / 8 / 4 neurons per expert
BASE_K        = 304                   # best from M3's sweep (reference point)
QUANTILE      = 0.5
TOKEN_CLUSTER_KS = [15, 40, 76]      # arm 2b: token clusters (== one per category at 15)
KMEANS_SEED   = 0
MI_PERMUTATIONS = 200                 # permutation test for MI significance

print(f"fine k={FINE_KS}; token clusters={TOKEN_CLUSTER_KS}; "
      f"permutations={MI_PERMUTATIONS}")

**What just happened:** knobs set. Next: reload M3's cached captures — no model forward passes needed.

## Step 3 — Load cached captures

45 (layer, task) matrices from M3, straight from disk. We also keep the
per-token task label for every row — that's what arm 2b needs.

In [ ]:
import numpy as np
from src.datasets import CATEGORIES

z_store, labels = {}, {}   # (layer, task) -> matrix ; (layer) -> [T] task labels
for lk in LAYERS:
    mats, labs = [], []
    for task in CATEGORIES:
        f = CACHE_DIR / f"{lk.replace('.', '_')}__{task}.npy"
        assert f.exists(), f"missing capture {f} — run neuron_clustering.ipynb step 4 first"
        Z = np.load(f)
        mats.append(Z)
        labs += [task] * Z.shape[0]
    z_store[lk] = np.concatenate(mats, axis=0)
    labels[lk] = np.array(labs)
    print(f"{lk}: {z_store[lk].shape[0]} tokens with task labels")

**What just happened:** captures loaded with per-token task labels. Next: arm 2a.

## Step 4 — ARM 2a: finer clustering (k down to 4 neurons/expert)

Two safeguards against fooling ourselves:
1. **Token-budget check** — with only 763 tokens per layer, a k=2432 clustering
   has clusters averaging <1 token worth of signal; entropy on such clusters is
   meaningless. We report cluster occupancy alongside entropy.
2. Same pooled threshold + normalized entropy as M3, so numbers are directly
   comparable (M3 reference: contiguous 1.85, clustered-76 1.86, all 0% specialized).

In [ ]:
import importlib, src.clustering
importlib.reload(src.clustering)
from src.clustering import cluster_neurons
from src.micro_experts import contiguous_partition, groups_from_assignment

def pooled_entropy(z_by_task, groups, quantile=0.5):
    """Pooled-threshold P(active|task) -> normalized entropy [E]."""
    energies = []
    for Z in z_by_task.values():
        e = np.asarray(Z, np.float32) ** 2
        energies.append(np.stack([e[:, g].mean(1) for g in groups], axis=1))
    allE = np.concatenate(energies, axis=0)
    thr = np.quantile(allE, quantile, axis=0, keepdims=True)
    P = np.stack([(E > thr).mean(0) for E in energies], axis=0)
    p = np.clip(P, 1e-12, 1)
    H = -(p * np.log2(p)).sum(0) / np.log2(P.shape[0])
    return H, P

arm2a = []
for lk in LAYERS:
    Z = z_store[lk]
    z_by_task = {t: Z[labels[lk] == t] for t in CATEGORIES}
    for k in [BASE_K] + FINE_KS:
        a = cluster_neurons(Z, n_clusters=k, method="kmeans", random_state=KMEANS_SEED)
        sizes = np.bincount(a)
        occ = float((sizes >= 3).mean())              # clusters with >=3 neurons... proxy
        # token-budget: median tokens per cluster is fixed by T/k; flag starved k
        H, _ = pooled_entropy(z_by_task, groups_from_assignment(a), QUANTILE)
        arm2a.append({"layer": lk, "k": k, "meanH": float(H.mean()),
                      "frac_specialized": float((H < 0.75).mean()),
                      "neurons_per_expert": int(Z.shape[1] // k),
                      "tokens_per_expert": round(Z.shape[0] / k, 2)})
        print(f"{lk} k={k:4d} ({Z.shape[1]//k:3d} neu/exp, "
              f"{Z.shape[0]/k:6.1f} tok/exp): meanH={H.mean():.3f} "
              f"spec={(H<0.75).mean():.0%}")

**What just happened:** the fine-k entropy curve is measured — including the tokens-per-expert budget that tells us where the statistics become meaningless. Next: arm 2b, the more important test.

## Step 5 — ARM 2b: contribution vectors per token

For every token: a 76-dim vector of **block contributions** ‖W_down·z_e‖ —
exactly the score M1's oracle ranked blocks by. This is the "routing
fingerprint" of a token. Needs the down_proj weights briefly (model must be
loaded — same kernel as M3 or a quick reload).

In [ ]:
import torch
from src.model_utils import load_model_and_tokenizer, describe_model, get_mlp, mlp_down_weight
from src.micro_experts import contiguous_partition, expert_importance_output

if "model" not in globals():
    model, tokenizer = load_model_and_tokenizer("Qwen/Qwen3-4B", "bfloat16")
    info = describe_model(model)

E76 = INTER // 128 if "INTER" in globals() else 76
contig76 = contiguous_partition(9728, 128)

contrib = {}   # layer -> [T, 76] contribution vectors
for lk in LAYERS:
    w = mlp_down_weight(model, lk, device=next(model.parameters()).device)
    Zt = torch.from_numpy(z_store[lk]).to(w.device)
    C = expert_importance_output(Zt, w, contig76)     # [T, 76]
    contrib[lk] = C.float().cpu().numpy().astype(np.float32)
    print(f"{lk}: contribution matrix {contrib[lk].shape}")

**What just happened:** every token now has a 76-dim routing fingerprint. Next: the decisive test.

## Step 6 — ARM 2b: do token fingerprints align with task categories?

Cluster the *tokens* by fingerprint (k = 15/40/76), then measure **mutual
information** between cluster id and task label, with a permutation test for
significance. High MI = routing patterns are task-informative (specialization
exists in routing space). MI ≈ 0 = a token's routing pattern says nothing
about its task — the strongest possible null, independent of any neuron
partition.

In [ ]:
def mutual_info(cluster_ids, task_labels):
    """MI(cluster; task) in bits, computed from the joint contingency."""
    import numpy as np
    ci = {c: i for i, c in enumerate(sorted(set(cluster_ids)))}
    ti = {t: i for i, t in enumerate(sorted(set(task_labels)))}
    J = np.zeros((len(ci), len(ti)))
    for c, t in zip(cluster_ids, task_labels):
        J[ci[c], ti[t]] += 1
    J /= J.sum()
    pc, pt = J.sum(1, keepdims=True), J.sum(0, keepdims=True)
    indep = pc @ pt
    mask = J > 0
    return float((J[mask] * np.log2(J[mask] / indep[mask])).sum())

rng = np.random.default_rng(KMEANS_SEED)
arm2b = []
for lk in LAYERS:
    X = contrib[lk]
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
    labs = labels[lk]
    H_task = mutual_info(labs, labs)   # = task entropy itself (upper bound)
    for k in TOKEN_CLUSTER_KS:
        a = cluster_neurons(Xn, n_clusters=k, method="kmeans",
                            random_state=KMEANS_SEED)
        mi = mutual_info(a.tolist(), labs.tolist())
        # permutation test: shuffle task labels, recompute MI
        null = []
        for _ in range(MI_PERMUTATIONS):
            perm = rng.permutation(len(labs))
            null.append(mutual_info(a.tolist(), labs[perm].tolist()))
        pval = float((np.array(null) >= mi).mean())
        arm2b.append({"layer": lk, "k": k, "MI_bits": mi,
                      "task_entropy_bits": H_task,
                      "frac_MI_explained": mi / max(H_task, 1e-9),
                      "p_value": pval})
        print(f"{lk} k={k:2d}: MI={mi:.3f} bits of {H_task:.3f} "
              f"({mi/max(H_task,1e-9):.0%} of task entropy), perm-p={pval:.3f}")

**What just happened:** the decisive numbers are in. Next: plots and the verdict.

## Step 7 — Plots, verdict, and archival

Everything (CSVs, PNGs, JSON) is written into
`results/m3b_granularity_option2/` and a line is appended to
`results/RUN_MANIFEST.md` — the append-only paper trail.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
# arm 2a: entropy vs granularity
for lk in LAYERS:
    pts = sorted([(r["k"], r["meanH"]) for r in arm2a if r["layer"] == lk])
    axes[0].plot([p[0] for p in pts], [p[1] for p in pts], marker="o",
                 label=lk.split(".")[-1])
axes[0].axhline(2.0 * 0.925, ls="--", c="gray", lw=1, label="spec threshold (H<1.85)")
axes[0].set_xscale("log"); axes[0].set_xlabel("cluster count k (log)")
axes[0].set_ylabel("mean task entropy"); axes[0].set_title("arm 2a: finer clusters")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
# arm 2b: MI explained
for lk in LAYERS:
    pts = sorted([(r["k"], r["frac_MI_explained"]) for r in arm2b if r["layer"] == lk])
    axes[1].plot([p[0] for p in pts], [p[1] for p in pts], marker="s",
                 label=lk.split(".")[-1])
axes[1].set_xlabel("token-cluster count k"); axes[1].set_ylabel("MI / task entropy")
axes[1].set_title("arm 2b: routing-fingerprint task information")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plot_path = M3B / "plots" / "option2_summary.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print("saved plot:", plot_path)

# CSVs
with open(M3B / "csv" / "arm2a_fine_k.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(arm2a[0].keys())); w.writeheader(); w.writerows(arm2a)
with open(M3B / "csv" / "arm2b_token_mi.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(arm2b[0].keys())); w.writeheader(); w.writerows(arm2b)
print("saved CSVs: arm2a_fine_k.csv, arm2b_token_mi.csv")

In [ ]:
best_spec_2a = max(r["frac_specialized"] for r in arm2a)
best_mi_2b = max(r["frac_MI_explained"] for r in arm2b)
best_p = min(r["p_value"] for r in arm2b)

verdict = {
    "run_stamp": RUN_STAMP,
    "arm2a_best_frac_specialized": best_spec_2a,
    "arm2b_best_frac_MI_explained": best_mi_2b,
    "arm2b_best_p_value": best_p,
}
with open(M3B / "json" / "option2_verdict.json", "w") as f:
    json.dump({"verdict": verdict, "arm2a": arm2a, "arm2b": arm2b}, f, indent=2)

print(f"arm 2a: best specialized fraction = {best_spec_2a:.0%}")
print(f"arm 2b: best MI explained = {best_mi_2b:.0%} of task entropy (p={best_p:.3f})")

if best_mi_2b > 0.30 and best_p < 0.05:
    print("\n-> TASK INFORMATION EXISTS in routing fingerprints! Revisit prompt-level routing before M4.")
elif best_mi_2b > 0.10 and best_p < 0.05:
    print("\n-> WEAK task information: significant but tiny. Token-level router (M4) remains the plan; note as caveat.")
else:
    print("\n-> CONFIRMED NULL: task specialization does not exist in routing space either. "
          "Pivot to token-level router (M4) per §55.7, with three rigorous nulls as evidence.")

# append to the run manifest (append-only)
with open(ROOT / "results" / "RUN_MANIFEST.md", "a", encoding="utf-8") as f:
    f.write(f"\n| {RUN_STAMP} | M3b option-2 | finer-k + token-MI arms | "
            f"2a spec={best_spec_2a:.0%}, 2b MI={best_mi_2b:.0%} (p={best_p:.3f}) | "
            f"notebooks/option2_granularity.ipynb | m3b_granularity_option2/ |\n")
print("manifest updated: results/RUN_MANIFEST.md")

**What just happened:** Option 2 is fully tested, archived, and the manifest updated.

## Deliverables (all in `results/m3b_granularity_option2/`)

| file | content |
|---|---|
| `csv/arm2a_fine_k.csv` | entropy at k=304..2432 with tokens-per-expert budget |
| `csv/arm2b_token_mi.csv` | MI between token-routing-fingerprints and tasks + permutation p-values |
| `plots/option2_summary.png` | both arms in one figure |
| `json/option2_verdict.json` | full machine-readable results |
| `results/RUN_MANIFEST.md` | appended run provenance line |

## Where this leaves the project

- If 2b shows real MI: prompt-level routing is back on the table.
- If null (expected per M2/M3): proceed to **M4 token-level router** — the paper
  story is "token-locally sparse, not task-partitioned", with this notebook as
  the final nail. Either way, M4 is next; only the framing changes.